# Assignment 4
Mohammad Rashiqul Alam  
malam25@illinois.edu  

# Part 1: Single-View Geometry

## Usage
This code snippet provides an overall code structure and some interactive plot interfaces for the *Single-View Geometry* section of Assignment 3. In [main function](#Main-function), we outline the required functionalities step by step. Some of the functions which involves interactive plots are already provided, but [the rest](#Your-implementation) are left for you to implement.

## Package installation
- In this code, we use `tkinter` package. Installation instruction can be found [here](https://anaconda.org/anaconda/tk).

# Common imports

In [1]:
%matplotlib tk
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from PIL import Image

# Provided functions

In [2]:
def get_input_lines(im, min_lines=3):
    """
    Allows user to input line segments; computes centers and directions.
    Inputs:
        im: np.ndarray of shape (height, width, 3)
        min_lines: minimum number of lines required
    Returns:
        n: number of lines from input
        lines: np.ndarray of shape (3, n)
            where each column denotes the parameters of the line equation
        centers: np.ndarray of shape (3, n)
            where each column denotes the homogeneous coordinates of the centers
    """
    n = 0
    lines = np.zeros((3, 0))
    centers = np.zeros((3, 0))

    plt.figure()
    plt.imshow(im)
    plt.show()
    print('Set at least %d lines to compute vanishing point' % min_lines)
    while True:
        print('Click the two endpoints, use the right key to undo, and use the middle key to stop input')
        clicked = plt.ginput(2, timeout=0, show_clicks=True)
        if not clicked or len(clicked) < 2:
            if n < min_lines:
                print('Need at least %d lines, you have %d now' % (min_lines, n))
                continue
            else:
                # Stop getting lines if number of lines is enough
                break

        # Unpack user inputs and save as homogeneous coordinates
        pt1 = np.array([clicked[0][0], clicked[0][1], 1])
        pt2 = np.array([clicked[1][0], clicked[1][1], 1])
        # Get line equation using cross product
        # Line equation: line[0] * x + line[1] * y + line[2] = 0
        line = np.cross(pt1, pt2)
        lines = np.append(lines, line.reshape((3, 1)), axis=1)
        # Get center coordinate of the line segment
        center = (pt1 + pt2) / 2
        centers = np.append(centers, center.reshape((3, 1)), axis=1)

        # Plot line segment
        plt.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]], color='b')

        n += 1

    return n, lines, centers

In [3]:
def plot_lines_and_vp(im, lines, vp):
    """
    Plots user-input lines and the calculated vanishing point.
    Inputs:
        im: np.ndarray of shape (height, width, 3)
        lines: np.ndarray of shape (3, n)
            where each column denotes the parameters of the line equation
        vp: np.ndarray of shape (3, )
    """
    bx1 = min(1, vp[0] / vp[2]) - 10
    bx2 = max(im.shape[1], vp[0] / vp[2]) + 10
    by1 = min(1, vp[1] / vp[2]) - 10
    by2 = max(im.shape[0], vp[1] / vp[2]) + 10

    plt.figure()
    plt.imshow(im)
    for i in range(lines.shape[1]):
        if lines[0, i] < lines[1, i]:
            pt1 = np.cross(np.array([1, 0, -bx1]), lines[:, i])
            pt2 = np.cross(np.array([1, 0, -bx2]), lines[:, i])
        else:
            pt1 = np.cross(np.array([0, 1, -by1]), lines[:, i])
            pt2 = np.cross(np.array([0, 1, -by2]), lines[:, i])
        pt1 = pt1 / pt1[2]
        pt2 = pt2 / pt2[2]
        plt.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]], 'g')

    plt.plot(vp[0] / vp[2], vp[1] / vp[2], 'ro')
    plt.show()

In [4]:
def get_top_and_bottom_coordinates(im, obj):
    """
    For a specific object, prompts user to record the top coordinate and the bottom coordinate in the image.
    Inputs:
        im: np.ndarray of shape (height, width, 3)
        obj: string, object name
    Returns:
        coord: np.ndarray of shape (3, 2)
            where coord[:, 0] is the homogeneous coordinate of the top of the object and coord[:, 1] is the homogeneous
            coordinate of the bottom
    """
    plt.figure()
    plt.imshow(im)

    print('Click on the top coordinate of %s' % obj)
    clicked = plt.ginput(1, timeout=0, show_clicks=True)
    x1, y1 = clicked[0]
    # Uncomment this line to enable a vertical line to help align the two coordinates
    # plt.plot([x1, x1], [0, im.shape[0]], 'b')
    print('Click on the bottom coordinate of %s' % obj)
    clicked = plt.ginput(1, timeout=0, show_clicks=True)
    x2, y2 = clicked[0]

    plt.plot([x1, x2], [y1, y2], 'b')

    return np.array([[x1, x2], [y1, y2], [1, 1]])

# Your implementation

In [5]:
def get_vanishing_point(lines):
    """
    Solves for the vanishing point from a set of user-input lines.
    Each column of `lines` is a homogeneous line [a,b,c]^T so that a*x + b*y + c = 0.
    We solve L^T v = 0 in the least-squares sense and return a normalized homogeneous v.
    Inputs:
        lines: (3, n) ndarray
    Returns:
        vp: (3,) ndarray  (homogeneous; vp[2] != 0)
    """
    # SVD on L^T to find the 1D nullspace
    # L^T v = 0  -> v is right-singular vector associated with smallest singular value
    U, S, Vt = np.linalg.svd(lines.T)
    vp = Vt[-1, :]
    # normalize so that vp[2] = 1 when possible
    if abs(vp[2]) > 1e-12:
        vp = vp / vp[2]
    return vp

In [6]:
def get_horizon_line(vp1, vp2):
    """
    Ground horizon line through the two horizontal vanishing points.
    Inputs:
        vp1, vp2: each (3,) homogeneous vanishing points with vp[2] possibly == 1
    Returns:
        h: (3,) homogeneous line parameters (a,b,c) with ||(a,b)|| = 1
    """
    # Ensure homogeneous scaling by making vp[2] = 1 when possible
    if abs(vp1[2]) > 1e-12: 
        vp1 = vp1 / vp1[2]
    if abs(vp2[2]) > 1e-12: 
        vp2 = vp2 / vp2[2]
    # line through the two VPs
    h = np.cross(vp1, vp2)       
    # Normalize so that a^2 + b^2 = 1
    ab_norm = np.linalg.norm(h[:2])
    # Avoid division by zero
    if ab_norm > 0:
        h = h / ab_norm
    return h

In [7]:
def plot_horizon_line(im, horizon_line):
    """
    Plots the horizon line over the image.
    Inputs:
        im: (H,W,3) image
        horizon_line: (3,) line parameters (a,b,c) with a^2 + b^2 = 1
    """
    a, b, c = horizon_line
    H, W = im.shape[0], im.shape[1]

    # Compute intersections with left/right borders (x=0 and x=W-1)
    # For b != 0, y = -(a*x + c)/b; handle near-vertical horizon robustly
    xs = np.array([0, W-1], dtype=float)
    ys = np.full_like(xs, np.nan, dtype=float)
    # 1e-12 is arbitrary small threshold to avoid division by zero
    if abs(b) > 1e-12:
        ys = -(a*xs + c)/b
    else:
        # vertical horizon (rare) -> use intersections with top/bottom to get x
        # a*x + b*y + c = 0 -> x = -c/a
        x_const = -c/a
        xs = np.array([x_const, x_const])
        ys = np.array([0, H-1], dtype=float)

    plt.figure()
    plt.imshow(im)
    plt.plot(xs, ys, 'r-', linewidth=2)
    plt.title('Horizon line')
    plt.show()

In [8]:
def get_camera_parameters(vpts, image_shape=None):
    """
    Solve for (f, u, v) using SymPy and the three orthogonal vanishing points.

    Inputs:
        vpts : (3,3) ndarray; columns are homogeneous vanishing points for 3 orthogonal world axes
        image_shape : optional (H, W) to seed u,v near image center and f near image size

    Returns:
        f, u, v  (floats)
    """
    V = vpts.astype(float).copy()
    # Normalize each VP to z=1 for numerical stability
    for i in range(3):
        if abs(V[2, i]) > 1e-12:
            V[:, i] /= V[2, i]

    # Symbols
    f, u, v = sp.symbols('f u v', real=True)

    # Intrinsics and IAC
    K = sp.Matrix([[f, 0, u],
                   [0, f, v],
                   [0, 0, 1]])
    Kinv = K.inv()
    omega = Kinv.T * Kinv  # Image of the Absolute Conic

    # Build equations v_i^T ω v_j = 0 for the 3 pairs
    Vsym = [sp.Matrix(V[:, i]) for i in range(3)]
    pairs = [(0,1), (0,2), (1,2)]
    eqs = []
    for i, j in pairs:
        eqs.append(sp.Eq((Vsym[i].T * omega * Vsym[j])[0], 0))

    # Initial guess
    if image_shape is not None:
        H, W = image_shape
        u0 = float(W) / 2.0
        v0 = float(H) / 2.0
        f0 = float(max(H, W))
    else:
        u0, v0, f0 = 800.0, 600.0, 1500.0

    # Use nsolve; try a couple of seeds if needed
    unknowns = (f, u, v)
    seeds = [
        (f0, u0, v0),
        (1.5*f0, u0*0.9, v0*1.1),
        (0.8*f0, u0*1.1, v0*0.9),
    ]
    sol = None
    for s in seeds:
        try:
            sol_candidate = sp.nsolve(eqs, unknowns, s, tol=1e-16, maxsteps=200)
            # Ensure positive focal length
            if float(sol_candidate[0]) > 0:
                sol = sol_candidate
                break
        except Exception:
            continue
    if sol is None:
        # As a last resort, allow negative root then flip sign
        sol = sp.nsolve(eqs, unknowns, seeds[0], tol=1e-16, maxsteps=200)

    f_val, u_val, v_val = map(float, sol)
    f_val = abs(f_val)  # enforce positive focal length
    return f_val, u_val, v_val

In [9]:
def get_rotation_matrix(f=None, u=None, v=None, vpts=None):
    """
    Computes the camera rotation matrix R given intrinsics and vanishing points.
    By definition, direction vectors r_i are proportional to K^{-1} v_i.
    We form columns r1,r2,r3 and orthonormalize by normalization + right-handed fix.
    Inputs:
        f,u,v: intrinsics
        vpts: (3,3) array with columns the three vanishing points
    Returns:
        R: (3,3) rotation matrix
    """
    assert vpts is not None and f is not None and u is not None and v is not None
    # Build K^{-1}
    Kinv = np.array([[1.0/f,     0.0,   -u/f],
                     [0.0,     1.0/f,   -v/f],
                     [0.0,       0.0,    1.0]])
    V = vpts.copy().astype(float)
    for i in range(3):
        if abs(V[2,i]) > 1e-12:
            V[:,i] /= V[2,i]

    Rcols = Kinv @ V  # columns are unnormalized directions
    # Normalize columns
    for i in range(3):
        n = np.linalg.norm(Rcols[:, i])
        if n > 1e-12:
            Rcols[:, i] /= n

    # Enforce orthonormality (small drift): r3 = r1 x r2, then re-orthonormalize
    r1 = Rcols[:,0]
    r2 = Rcols[:,1]
    r3 = np.cross(r1, r2)
    r2 = np.cross(r3, r1)
    r1 /= np.linalg.norm(r1); r2 /= np.linalg.norm(r2); r3 /= np.linalg.norm(r3)
    R = np.stack([r1, r2, r3], axis=1)
    return R

In [10]:
def estimate_height(coord_obj, coord_ref, v_vert, horizon_line, ref_height_m,
                    make_plot=False, im=None, obj_name=""):
    """
    Estimate an object's metric height using a known reference person and the
    vertical vanishing point + horizon line (Criminisi et al.).

    Inputs:
        coord_obj : (3,2) homogeneous pts -> [:,0]=top, [:,1]=bottom of target
        coord_ref : (3,2) homogeneous pts -> [:,0]=top, [:,1]=bottom of reference
        v_vert    : (3,)  vertical vanishing point (homogeneous)
        horizon_line : (3,) ground horizon line (a,b,c) with a^2+b^2=1
        ref_height_m : float, known reference height in meters
        make_plot: bool, optionally draw the construction
        im       : image array, required if make_plot=True
        obj_name : str, label for the plot

    Returns:
        height_m : float, estimated height in meters
    """
    def deh(p):
        return p[:2] / p[2]

    # Ensure homogeneous normalization
    v = v_vert.astype(float)
    if abs(v[2]) > 1e-12:
        v = v / v[2]

    # Lines through the bottoms toward the vertical VP
    L_obj = np.cross(coord_obj[:, 1], v)   # line through object base and vertical VP
    L_ref = np.cross(coord_ref[:, 1], v)   # line through ref base and vertical VP

    # Intersections with horizon: I = ( (base × Vz) × horizon )
    I_obj = np.cross(L_obj, horizon_line)
    I_ref = np.cross(L_ref, horizon_line)

    # Dehomogenize all needed points
    xT, xB = deh(coord_obj[:, 0]), deh(coord_obj[:, 1])
    rT, rB = deh(coord_ref[:, 0]), deh(coord_ref[:, 1])
    I_obj = deh(I_obj)
    I_ref = deh(I_ref)

    # Euclidean distances in image
    def dist(a, b): return float(np.linalg.norm(a - b))
    d_xx   = dist(xT, xB)     # |x x'|
    d_xI   = dist(xB, I_obj)  # |x I|
    d_rI   = dist(rB, I_ref)  # |x0 I0|
    d_rr   = dist(rT, rB)     # |x0 x0'|

    # Cross-ratio based height transfer:
    # H = h * ( |x x'| / |x I| ) * ( |x0 I0| / |x0 x0'| )
    eps = 1e-12
    height_m = ref_height_m * (d_xx / max(d_xI, eps)) * (d_rI / max(d_rr, eps))

    if make_plot and im is not None:
        plt.figure()
        plt.imshow(im)
        # draw vertical rays
        def draw_ray(base):
            # extend to image bounds using intersections with frame
            H, W = im.shape[0], im.shape[1]
            # intersect with left/right borders to make a visible segment
            # line: L_obj or L_ref depending on base
        plt.plot([xB[0], I_obj[0]], [xB[1], I_obj[1]], 'g--', lw=1)
        plt.plot([rB[0], I_ref[0]], [rB[1], I_ref[1]], 'c--', lw=1)
        # segments
        plt.plot([xB[0], xT[0]], [xB[1], xT[1]], 'y-', lw=2, label=f'{obj_name} segment')
        plt.plot([rB[0], rT[0]], [rB[1], rT[1]], 'm-', lw=2, label='reference segment')
        # points
        plt.plot([I_obj[0]], [I_obj[1]], 'ro', label='I (obj)')
        plt.plot([I_ref[0]], [I_ref[1]], 'bo', label='I (ref)')
        plt.title(f'Height construction: {obj_name}  ≈  {height_m:.2f} m')
        plt.legend()
        plt.show()

    return height_m

# Main function

In [ ]:
# Load image
im = np.asarray(Image.open('CSL.jpg'))

# Part 1
# Get vanishing points for each of the directions
num_vpts = 3
vpts = np.zeros((3, num_vpts))
saved_lines = []   # keep each direction's lines for plotting/report

for i in range(num_vpts):
    print('Getting vanishing point %d' % i)
    # Get at least three lines from user input
    n, lines, centers = get_input_lines(im)
    saved_lines.append(lines)
    # <YOUR IMPLEMENTATION> Solve for vanishing point
    # Compute vanishing point via SVD
    vpts[:, i] = get_vanishing_point(lines)
    # Plot the lines and the vanishing point
    plot_lines_and_vp(im, lines, vpts[:, i])

# <YOUR IMPLEMENTATION> Get the ground horizon line
horizon_line = get_horizon_line(vpts[:, 0], vpts[:, 1])
# <YOUR IMPLEMENTATION> Plot the ground horizon line
plot_horizon_line(im, horizon_line)

# Part 2
# <YOUR IMPLEMENTATION> Solve for the camera parameters (f, u, v)
f, u, v = get_camera_parameters(vpts, im.shape[:2])
# Part 3
# <YOUR IMPLEMENTATION> Solve for the rotation matrix
R = get_rotation_matrix(f=f, u=u, v=v, vpts=vpts)

# Part 4
# Record image coordinates for each object and store in map
objects = ('person', 'CSL building', 'the spike statue', 'the lamp posts')
coords = dict()
for obj in objects:
    coords[obj] = get_top_and_bottom_coordinates(im, obj)

# <YOUR IMPLEMENTATION> Estimate heights
ref_name = 'person'
ref_height_m = 66 * 0.0254        # 5 ft 6 in  -> meters (change to 72*0.0254 for 6 ft)
v_vert = vpts[:, 2]               # vertical vanishing point

# Initialize a dictionary to store calculated heights
heights = {}

# Loop through all target objects (skip the reference)
for obj in objects[1:]:
    print(f'Estimating height of {obj}')
    height = estimate_height(coords[obj],
                             coords[ref_name],
                             v_vert,
                             horizon_line,
                             ref_height_m,
                             make_plot=True, im=im, obj_name=obj)
    heights[obj] = height   # save each computed height
    print(f'  {obj}: {height:.2f} m  (with {ref_height_m/0.0254:.0f}" reference)')


Getting vanishing point 0
Set at least 3 lines to compute vanishing point
Click the two endpoints, use the right key to undo, and use the middle key to stop input


can't invoke "event" command: application has been destroyed
    while executing
"event generate $w <<ThemeChanged>>"
    (procedure "ttk::ThemeChanged" line 6)
    invoked from within
"ttk::ThemeChanged"


In [ ]:
# Summary section
print("\n================== REPORT SUMMARY ==================")

# Vanishing points (pixel coordinates)
print("Vanishing points (pixels):")
VPs_px = (vpts / vpts[2])[:2].T  # normalize each so z=1, take x,y only
for i, (x, y) in enumerate(VPs_px):
    print(f"  VP{i}: ({x:.2f}, {y:.2f})")

# Horizon line parameters
a, b, c = horizon_line
print("\nGround horizon line (normalized so a^2 + b^2 = 1):")
print(f"  a*x + b*y + c = 0")
print(f"  a = {a:.6f}, b = {b:.6f}, c = {c:.6f}")
print(f"  (a^2 + b^2) = {a*a + b*b:.3f}")

# Camera parameters
print("\nCamera intrinsic parameters:")
print(f"  f = {f:.3f} px")
print(f"  u = {u:.3f} px")
print(f"  v = {v:.3f} px")

# Rotation matrix
print("\nRotation matrix R:")
np.set_printoptions(precision=6, suppress=True)
print(R)

# Object heights using 5'6" reference
print("\nEstimated heights (reference = 5 ft 6 in):")
for obj, h in heights.items():
    print(f"  {obj}: {h:.2f} m ({h*3.28084:.1f} ft)")

# Recalculate if reference = 6 ft
scale = 72 / 66
print("\nIf reference person is 6 ft tall:")
for obj, h in heights.items():
    h_scaled = h * scale
    print(f"  {obj}: {h_scaled:.2f} m ({h_scaled*3.28084:.1f} ft)")

print("====================================================\n")
